In [ ]:
!pip install ucimlrepo imbalanced-learn shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_recall_fscore_support)
from imblearn.over_sampling import SMOTE
import joblib

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Libraries imported successfully!")

In [ ]:
from ucimlrepo import fetch_ucirepo

steel_plates = fetch_ucirepo(id=198)

X = steel_plates.data.features
y_multi = steel_plates.data.targets

print("Dataset loaded!")
print("Shape of features:", X.shape)
print("7 binary target columns:", y_multi.columns.tolist())

In [ ]:
# Create single target column from 7 binary columns
y = y_multi.idxmax(axis=1)
X = X.copy()

print("Target created successfully!")
print("Unique defect types (7 classes):")
print(y.value_counts().sort_index())

In [ ]:
plt.figure(figsize=(10,5))
y.value_counts().plot(kind='bar', color='skyblue')
plt.title('Distribution of 7 Defect Types')
plt.xlabel('Defect Type')
plt.ylabel('Number of Plates')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nClass distribution:\n", y.value_counts(normalize=True).round(3))

In [ ]:
# Label Encoding
le = LabelEncoder()
y_numeric = le.fit_transform(y)

mapping = dict(zip(le.classes_, range(len(le.classes_))))
print("Label mapping:")
for defect, num in sorted(mapping.items()):
    print(f"  {num:2d} ← {defect}")

y = y_numeric
print("\nTarget is now numeric. Unique values:", np.unique(y))

In [ ]:
# Train/Test Split  ← must happen BEFORE SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size:  {X_test.shape[0]} rows")

In [ ]:
# Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("After SMOTE: ", pd.Series(y_train_smote).value_counts().sort_index().to_dict())
print(f"Training samples: {len(X_train)} → {len(X_train_smote)}")

In [ ]:
# Preprocessor: StandardScaler on all features
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), X.columns.tolist())
])

In [ ]:
# Define 3 model pipelines
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(multi_class='multinomial',
        solver='lbfgs', max_iter=3000, random_state=42))
])

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=400, max_depth=15, random_state=42, n_jobs=-1))
])

xgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='multi:softprob', num_class=7,
        max_depth=8, learning_rate=0.08, n_estimators=400,
        subsample=0.85, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, eval_metric='mlogloss'))
])

print("✅ 3 pipelines defined!")

In [ ]:
# Train without SMOTE
print("Training baseline models...\n")
lr_pipe.fit(X_train, y_train);  print("Logistic Regression → Done")
rf_pipe.fit(X_train, y_train);  print("Random Forest       → Done")
xgb_pipe.fit(X_train, y_train); print("XGBoost             → Done")

In [ ]:
def evaluate_model_full(pipe, name, X_test=X_test, y_test=y_test, target_names=None):
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred,
        labels=range(len(target_names)) if target_names else None,
        zero_division=0
    )
    print(f"\n{'='*70}")
    print(f"Evaluation: {name}")
    print(f"{'='*70}")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Macro Precision: {precision.mean():.4f}")
    print(f"Macro Recall:    {recall.mean():.4f}")
    print(f"Macro F1-score:  {f1.mean():.4f}\n")
    if target_names:
        print("Class".ljust(18) + "Precision".rjust(12) + "Recall".rjust(10) + "F1".rjust(10) + "Support".rjust(10))
        print("-" * 60)
        for i, cls in enumerate(target_names):
            print(f"{cls:<18} {precision[i]:>12.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {support[i]:>10}")
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names, cbar=False)
    plt.title(f'Confusion Matrix – {name}\nAccuracy: {acc:.4f}', fontsize=14, pad=20)
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()

In [ ]:
class_names = le.classes_.tolist()

evaluate_model_full(lr_pipe,  "Logistic Regression", target_names=class_names)
evaluate_model_full(rf_pipe,  "Random Forest",        target_names=class_names)
evaluate_model_full(xgb_pipe, "XGBoost",              target_names=class_names)

In [ ]:
# Define SMOTE pipelines (fresh pipelines needed)
lr_smote = Pipeline([
    ('preprocessor', ColumnTransformer([('num', StandardScaler(), X.columns.tolist())])),
    ('classifier', LogisticRegression(multi_class='multinomial',
        solver='lbfgs', max_iter=3000, random_state=42))
])

rf_smote = Pipeline([
    ('preprocessor', ColumnTransformer([('num', StandardScaler(), X.columns.tolist())])),
    ('classifier', RandomForestClassifier(
        n_estimators=400, max_depth=15, random_state=42, n_jobs=-1))
])

xgb_smote = Pipeline([
    ('preprocessor', ColumnTransformer([('num', StandardScaler(), X.columns.tolist())])),
    ('classifier', XGBClassifier(
        objective='multi:softprob', num_class=7,
        max_depth=8, learning_rate=0.08, n_estimators=400,
        subsample=0.85, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, eval_metric='mlogloss'))
])

print("✅ 3 SMOTE pipelines ready!")

In [ ]:
# Train with SMOTE data
print("Training SMOTE models...\n")
lr_smote.fit(X_train_smote, y_train_smote);  print("Logistic Regression + SMOTE → Done")
rf_smote.fit(X_train_smote, y_train_smote);  print("Random Forest + SMOTE       → Done")
xgb_smote.fit(X_train_smote, y_train_smote); print("XGBoost + SMOTE             → Done")

In [ ]:
class_names = ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']

print("=== Results After Applying SMOTE ===\n")
evaluate_model_full(lr_smote,  "Logistic Regression + SMOTE", target_names=class_names)
evaluate_model_full(rf_smote,  "Random Forest + SMOTE",       target_names=class_names)
evaluate_model_full(xgb_smote, "XGBoost + SMOTE",             target_names=class_names)

In [ ]:
print("\nFinal Comparison: Before vs After SMOTE")
print("-"*75)
print(f"{'Model':<30} {'Accuracy':>10} {'Macro F1':>12}")
print("-"*75)

for name, pipe in [("Logistic Regression", lr_pipe), ("Random Forest", rf_pipe), ("XGBoost", xgb_pipe)]:
    y_pred = pipe.predict(X_test)
    print(f"{name:<30} {accuracy_score(y_test, y_pred):>10.4f} {f1_score(y_test, y_pred, average='macro', zero_division=0):>12.4f}")

print("-"*75)
for name, pipe in [("Logistic+SMOTE", lr_smote), ("RF+SMOTE", rf_smote), ("XGBoost+SMOTE ⭐", xgb_smote)]:
    y_pred = pipe.predict(X_test)
    print(f"{name:<30} {accuracy_score(y_test, y_pred):>10.4f} {f1_score(y_test, y_pred, average='macro', zero_division=0):>12.4f}")

In [ ]:
y_pred_lr  = lr_smote.predict(X_test)
y_pred_rf  = rf_smote.predict(X_test)
y_pred_xgb = xgb_smote.predict(X_test)

class_names = ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']

# Actual vs Predicted for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, (mname, y_pred) in zip(axes, [("Logistic Regression", y_pred_lr),
                                       ("Random Forest", y_pred_rf),
                                       ("XGBoost", y_pred_xgb)]):
    actual_counts    = pd.Series(y_test).value_counts().sort_index()
    predicted_counts = pd.Series(y_pred).value_counts().sort_index()
    ax.plot(class_names, actual_counts.values,    marker='o', label='Actual',    color='blue')
    ax.plot(class_names, predicted_counts.values, marker='s', label='Predicted', color='orange')
    ax.set_title(mname); ax.set_xlabel('Defect Type'); ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Actual vs Predicted Defect Counts — All Models', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance (XGBoost)
importances = xgb_smote.named_steps['classifier'].feature_importances_
fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10,8))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Features Causing Defects (XGBoost + SMOTE)')
plt.tight_layout(); plt.show()
print(fi_df.to_string(index=False))

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_smote.named_steps['classifier'])
X_test_transformed = xgb_smote.named_steps['preprocessor'].transform(X_test)
shap_values = explainer.shap_values(X_test_transformed)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_transformed,
                  feature_names=X.columns.tolist(), max_display=15)
plt.title("SHAP Feature Impact on Defect Classification")
plt.show()

In [ ]:
# ============================================================
# SAVE MODEL COMPONENTS SEPARATELY
# This avoids sklearn version mismatch errors on local machines
# ============================================================
import joblib
from google.colab import files
import sklearn, xgboost as xgb_lib
print(f"Saving with: sklearn {sklearn.__version__} | xgboost {xgb_lib.__version__}")

# 1. Save XGBoost classifier only (no sklearn wrapper)
joblib.dump(xgb_smote.named_steps['classifier'], 'xgb_classifier.pkl')
print("✅ xgb_classifier.pkl saved")

# 2. Save the fitted scaler (StandardScaler fitted values)
fitted_scaler = xgb_smote.named_steps['preprocessor'].transformers_[0][1]
joblib.dump(fitted_scaler, 'scaler.pkl')
print("✅ scaler.pkl saved")

# 3. Save label encoder
joblib.dump(le, 'label_encoder.pkl')
print("✅ label_encoder.pkl saved")

# 4. Save feature names list
joblib.dump(X.columns.tolist(), 'feature_names.pkl')
print("✅ feature_names.pkl saved")

print("\n--- Downloading all files ---")
files.download('xgb_classifier.pkl')
files.download('scaler.pkl')
files.download('label_encoder.pkl')
files.download('feature_names.pkl')
print("✅ All downloads triggered!")